$Qa)$

In [ ]:
# This is code we have been provided and asked to use. NOT OWN WORK

import zipfile
from pprint import pprint
import numpy as np

def load_complex_npz(file):
    data = np.load(file)
    cells = data["cells"]
    coordinates = data["coordinates"]
    return (cells, coordinates)


# Load a complex from within the .zip
def load_complex_zip(zipname, complexname):
 with zipfile.ZipFile(zipname) as z:
    with z.open(complexname) as f:
        return load_complex_npz(f)
    

# Print out files inside the .zip
with zipfile.ZipFile("complexes.zip") as z:
 filenames = z.namelist()

$QA.1)$

In [ ]:
def split_cell(cell: np.ndarray[tuple[int], int], dimension: int):
    '''Takes a cell and splits it into it's simplices by dimension.

    Args:
        cell: The cell we are splitting into its component simplicies.
        dimension: The dimension of the cell.

    Yields: A list with list[dim]=a set of tuples of the simplicies of our cell.'''

    set_of_i_negative_1_dimensional_simplicies: set[tuple[int]] = set([tuple()])
    vertices: set[int] = set(cell)
    # Each Loop takes our list of simplicies of a number of dimensions and then creates a list with our vertex added to our tuple.
    for i in range (dimension):
        set_of_i_dimensional_simplicies: set[tuple[int]] = set() 
        for vertex in vertices:
            vertex = int(vertex)
            for i_negative_1_dimensional_simplex in set_of_i_negative_1_dimensional_simplicies:  
                if vertex not in i_negative_1_dimensional_simplex: 
                    set_of_i_dimensional_simplicies.add(tuple(sorted(i_negative_1_dimensional_simplex + (vertex,))))
        yield set_of_i_dimensional_simplicies
        set_of_i_negative_1_dimensional_simplicies = set_of_i_dimensional_simplicies


def build_complex(cells: np.ndarray[tuple[int, int], int]) -> list[dict[tuple, int]]:
    '''Turns our Complex as a list of cells into a list of dictionaries with each dictionary containig all the simplicies of that dimension in the complex.
    
    Args:
        Cells: A list of all the highes dimension simplicies (cells) in our complex
        
    Returns: A list of dictionaries where list[dim] is a dictionary where all the positively oriented simplicies of dimension dim are a key for simplex Ids.'''
    
    dimension = (cells[0].shape)[0] # Gets the dimension of the first cell provided.
    _complex = [set()] * dimension 
    
    # Takes the union along each dimension of our list of simplicies for each cell to create a list of simplicies for the whole complex. 
    for cell in cells:
        split_up_cell = split_cell(cell,dimension)
        for i, _set in enumerate (split_up_cell):
            _complex[i] = sorted(_set.union(set(_complex[i])))

    # Turns our list of tuples into our required dictionary.
    complex = [dict()] * dimension
    for i, list_dimensional_simplicies in enumerate (_complex):
        i_dimensional_simplicies_dict = dict()
        for j, simplex in enumerate(list_dimensional_simplicies):
            i_dimensional_simplicies_dict[simplex]=j
        complex[i] = i_dimensional_simplicies_dict
    return complex


In [ ]:
# This is code we have been provided and asked to use as a diagnostic. NOT OWN WORK.

complex=build_complex(load_complex_zip('complexes.zip' ,'triangle.npz')[0])
for (dim, c) in enumerate(complex):
 print(f"Printing complex of dimension {dim}")
 pprint(c)

complex=build_complex(load_complex_zip('complexes.zip' ,'triangle_minus_triangle.npz')[0])
for (dim, c) in enumerate(complex):
 print(f"Printing complex of dimension {dim}")
 pprint(c)

Printing complex of dimension 0
{(0,): 0, (1,): 1, (2,): 2}
Printing complex of dimension 1
{(0, 1): 0, (0, 2): 1, (1, 2): 2}
Printing complex of dimension 2
{(0, 1, 2): 0}
Printing complex of dimension 0
{(0,): 0, (1,): 1, (2,): 2, (3,): 3, (4,): 4, (5,): 5}
Printing complex of dimension 1
{(0, 1): 0,
 (0, 2): 1,
 (0, 3): 2,
 (0, 5): 3,
 (1, 2): 4,
 (1, 3): 5,
 (1, 4): 6,
 (2, 4): 7,
 (2, 5): 8,
 (3, 4): 9,
 (3, 5): 10,
 (4, 5): 11}
Printing complex of dimension 2
{(0, 1, 3): 0,
 (0, 2, 5): 1,
 (0, 3, 5): 2,
 (1, 2, 4): 3,
 (1, 3, 4): 4,
 (2, 4, 5): 5}


$Qa.2)$

In [25]:
def boundary_matrix(complex: list[dict[tuple, int]], k: int) -> np.ndarray[tuple[int, int]]:
    ''' Creates the Matrix representation of our boundary linear transformation for δ_k.
    
    Args:
        complex: The complex to find the boundary linear transformation of.
        k: Dimension of simplicies after applying the δ_k boundary operator.

    Returns: The Matrix transformation, columns and rows correspond to the key in our complex dictionary.'''
    
    matrix = np.zeros((len(complex[k-1]), len(complex[k])))
    for col, cell in enumerate(complex[k]):
        for i in range(len(cell)):
            matrix[(complex[k-1][cell[:i] + cell[i+1:]])][col] = (-1)**i
    return matrix
            

for i in range (2):
    print (f'For triangle.npz, δ_{i+1} = \n {boundary_matrix(build_complex(load_complex_zip('complexes.zip', 'triangle.npz')[0]), i+1)}')
for i in range (2):
    print (f'For triangle_minus_triangle.npz, δ_{i+1} = \n {boundary_matrix(build_complex(load_complex_zip('complexes.zip', 'triangle_minus_triangle.npz')[0]), i+1)}')

For triangle.npz, δ_1 = 
 [[-1. -1.  0.]
 [ 1.  0. -1.]
 [ 0.  1.  1.]]
For triangle.npz, δ_2 = 
 [[ 1.]
 [-1.]
 [ 1.]]
For triangle_minus_triangle.npz, δ_1 = 
 [[-1. -1. -1. -1.  0.  0.  0.  0.  0.  0.  0.  0.]
 [ 1.  0.  0.  0. -1. -1. -1.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  1.  0.  0. -1. -1.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.  1.  0.  0.  0. -1. -1.  0.]
 [ 0.  0.  0.  0.  0.  0.  1.  1.  0.  1.  0. -1.]
 [ 0.  0.  0.  1.  0.  0.  0.  0.  1.  0.  1.  1.]]
For triangle_minus_triangle.npz, δ_2 = 
 [[ 1.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.]
 [-1.  0.  1.  0.  0.  0.]
 [ 0. -1. -1.  0.  0.  0.]
 [ 0.  0.  0.  1.  0.  0.]
 [ 1.  0.  0.  0.  1.  0.]
 [ 0.  0.  0. -1. -1.  0.]
 [ 0.  0.  0.  1.  0.  1.]
 [ 0.  1.  0.  0.  0. -1.]
 [ 0.  0.  0.  0.  1.  0.]
 [ 0.  0.  1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  1.]]


$Qa.3)$

In [ ]:
for npz_complex in filenames:
    # Sets up our complex.
    cells = load_complex_zip('complexes.zip', npz_complex)[0]
    dimension = (cells[0].shape)[0]
    complex = build_complex(cells)
    
    # Finds δ_i and δ_{i+1} matrix representation for our complex for all the relevant values.
    δ_i=boundary_matrix(build_complex(load_complex_zip('complexes.zip', npz_complex)[0]), 1)
    for i in range (1, dimension-1):
        δ_i_plus_1 = boundary_matrix(build_complex(load_complex_zip('complexes.zip', npz_complex)[0]), i+1)
        # Checks how many non zero entries and outputs it nicely.
        print (f'For {npz_complex}, δ_{i} ∘ δ_{i+1}  has {np.count_nonzero(np.matmul(δ_i, δ_i_plus_1))} non-0-entries.')
        δ_i = δ_i_plus_1 

For triangle.npz, δ_1 ∘ δ_2  has 0 non-0-entries.
For triangle_minus_triangle.npz, δ_1 ∘ δ_2  has 0 non-0-entries.
For hollow_sphere.npz, δ_1 ∘ δ_2  has 0 non-0-entries.
For solid_sphere.npz, δ_1 ∘ δ_2  has 0 non-0-entries.
For solid_sphere.npz, δ_2 ∘ δ_3  has 0 non-0-entries.
For two_spheres.npz, δ_1 ∘ δ_2  has 0 non-0-entries.
For two_spheres.npz, δ_2 ∘ δ_3  has 0 non-0-entries.
For sphere_minus_tunnel.npz, δ_1 ∘ δ_2  has 0 non-0-entries.
For sphere_minus_tunnel.npz, δ_2 ∘ δ_3  has 0 non-0-entries.
For sphere_minus_void.npz, δ_1 ∘ δ_2  has 0 non-0-entries.
For sphere_minus_void.npz, δ_2 ∘ δ_3  has 0 non-0-entries.
For unknown.npz, δ_1 ∘ δ_2  has 0 non-0-entries.
For unknown.npz, δ_2 ∘ δ_3  has 0 non-0-entries.


$Qa.4)$
$b_k = Nul(\delta_k) - Rank(\delta_{k+1})$

$b_k = Dim(C_k(K)) - Rank(\delta_k) - Rank(\delta_{k+1})$. By the Rank Nullity Theorem.

$b_k = ($ number of k simplices in K $) - Rank(\delta_k)- Rank(\delta_{k+1})$. By substituting in the value for $ Dim(C_k(K)) $.


$qa.5)$

In [ ]:
def rank_δ(complex: list[dict[tuple, int]], k: int , dimension: int) -> int:
    '''Finds the rank of the linear transformation of δ
    
    Args:
        Complex: The complex our linear transformation acts on.
        k: The dimension of the output of our linear combination.
        dimension: The dimension of our complex.
        
    Returns: The rank of our linear transformation.'''
    
    if k == 0 or k == dimension:
        return 0
    else:
        return np.linalg.matrix_rank(boundary_matrix(complex, k))


def betti_number (cells: np.ndarray[tuple[int, int], int]):
    ''' Finds the Betti number of our complex.
    
    Args:
        cells: The cells in our complex.
        
    Yields: the kth Betti number of our complex'''
    
    dimension = (cells[0].shape)[0]
    complex = build_complex(cells)
    rank_δ_k = rank_δ(complex, 0, dimension)
    for k in range (dimension):
        rank_δ_k_plus_one = rank_δ(complex, k+1, dimension)
        yield int(len(complex[k]) - rank_δ_k - rank_δ_k_plus_one)
        rank_δ_k = rank_δ_k_plus_one

for npz_complex in filenames:
    Betti_tuple = tuple(betti_number(load_complex_zip('complexes.zip', npz_complex)[0]))
    print(f'The Betti numbers of {npz_complex} are ({', '.join(map(lambda i : f'b_{i}', range(len(Betti_tuple))))}) = {Betti_tuple}.')
print('From this we can suggest unknown.npz is either (2 seperate spheres with a single void each) or (one sphere with 2 voids and a seperate solid sphere).') 

The Betti numbers of triangle.npz are (b_0, b_1, b_2) = (1, 0, 0).
From this we can suggest unknown.npz is either (2 seperate spheres with singular a void each) or (one sphere with 2 voids and a seperate solid sphere).
The Betti numbers of triangle_minus_triangle.npz are (b_0, b_1, b_2) = (1, 1, 0).
From this we can suggest unknown.npz is either (2 seperate spheres with singular a void each) or (one sphere with 2 voids and a seperate solid sphere).
The Betti numbers of hollow_sphere.npz are (b_0, b_1, b_2) = (1, 0, 1).
From this we can suggest unknown.npz is either (2 seperate spheres with singular a void each) or (one sphere with 2 voids and a seperate solid sphere).
The Betti numbers of solid_sphere.npz are (b_0, b_1, b_2, b_3) = (1, 0, 0, 0).
From this we can suggest unknown.npz is either (2 seperate spheres with singular a void each) or (one sphere with 2 voids and a seperate solid sphere).
The Betti numbers of two_spheres.npz are (b_0, b_1, b_2, b_3) = (2, 0, 0, 0).
From this we c